# Documentación del chatbot Fixify: Meta y PostgreSQL

El plan fue contrastado con los tres PDF originales el 4 de octubre de 2026. El código mostrado es un prototipo: el acceso PostgreSQL directo requiere adaptación para cumplir la arquitectura Backend/API de los documentos.


# Preparar PostgreSQL y ejecutar

La aplicación necesita una instancia PostgreSQL accesible. El código no instala ni arranca el servidor PostgreSQL. Incluso `CHATBOT_MODE=demo` usa PostgreSQL para eventos.

En `psql`, como administrador del servidor, crea un rol y base dedicados (ejecuta manualmente; usa una contraseña propia):

```sql
CREATE ROLE fixify_bot LOGIN PASSWORD 'REEMPLAZAR_POR_UN_SECRETO';
CREATE DATABASE fixify_chatbot OWNER fixify_bot;
```

El rol dueño de esta base puede crear las tablas y funciones del adaptador. El paquete 1.4.2 usa `public.history`, `public.contact` y las funciones `save_or_update_contact` y `save_or_update_history_and_contact`; no crea la base ni el rol. No requiere acceso a órdenes del negocio.

```bash
cd /home/deadtales/Escuela/Fixify/Fixify/chatbot
pnpm install --frozen-lockfile
pnpm build
cd base-ts-meta-postgres
# Si ya existe .env, agrega las variables nuevas sin sobrescribir tus credenciales.
# En una instalación nueva: cp .env.example .env
pnpm start
```

Configura `POSTGRES_HOST`, `POSTGRES_PORT`, `POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`, las variables de Meta y el modo. El arranque espera conexión y preparación de tablas antes de abrir HTTP. No cambia a MemoryDB si PostgreSQL falla. La configuración tipada básica del adaptador no expone SSL; una conexión remota requiere acordar la configuración segura soportada, no desactivar verificación de certificados.


# Plan de proyecto del chatbot Fixify: Bot, Backend y Frontend

Plan basado en los tres PDF originales, leídos el 4 de octubre de 2026: Product_Backlog_y_Requisitos_Sistema_Mantenimiento.pdf (§5.5–5.7 y reglas de negocio), Requerimientos del sistema.pdf (§4.5–4.7, §5.5–5.7 y §7), IEEE830_Fixify.pdf (§1.2, §2.1, §3.2 y §3.4). Los documentos son fuentes de requisitos; este notebook propone cómo implementarlos. Las tareas y contratos son propuestas técnicas, no requisitos textuales adicionales.



## 1. Objetivo y límites

Entregar un chatbot con Meta que consulte reparaciones y preguntas frecuentes mediante Backend, acompañado por las funciones necesarias de administración y notificaciones en el Frontend existente. El sistema conservará una única lógica de negocio en Backend y PostgreSQL central.

Esta etapa incluye HU-16–HU-22, RF-17–RF-19 y RN-08–RN-11. HU-22 tiene prioridad baja en el backlog, pero se incluye en el plan de alcance solicitado; la prioridad define el orden, no se usa para darla por implementada.

No se incluye editor de flujos, creación de comportamientos nuevos, edición arbitraria de disparadores, IA generativa ni administración de código. Las palabras que activan los flujos pueden permanecer en TypeScript. El administrador actualiza contenido FAQ y periodos de recordatorio; esto no exige editar la estructura del diálogo.



## 2. Arquitectura que cumple los documentos

```mermaid
flowchart LR
    A[Administrador] --> F[Frontend de escritorio]
    F --> B[Backend: autenticación, permisos y reglas]
    C[Cliente WhatsApp] --> M[Meta]
    M --> T[Bot BuilderBot]
    T --> B
    B --> D[(PostgreSQL central)]
    S[Proceso programado Backend] --> B
    B --> E[Solicitud de envío autorizada]
    E --> T
    T --> M
```

**Fuente:** IEEE §1.2 y §2.1; RF-19 y RN-11. El chatbot no consulta directamente PostgreSQL. Configuración, preguntas/respuestas, órdenes y registros de notificación pertenecen al Backend y su base central.

**Corrección necesaria del prototipo:** `src/database/postgres.ts` conecta directamente con PostgreSQLAdapter. Mantenerlo conectado a la base central seguiría contradiciendo el acceso mediante Backend. Para persistir eventos/contactos de BuilderBot sin romper el requisito, implementar un adaptador HTTP compatible con BuilderBot y endpoints internos de persistencia en Backend. PostgreSQL sigue siendo la base del sistema, pero sus credenciales permanecen en Backend. El historial conversacional es una propuesta de soporte técnico; no sustituye los eventos de negocio ni se presenta como un requisito funcional independiente.



## 3. Matriz de trabajo y aceptación

| Función / requisito | Bot | Backend y PostgreSQL | Frontend | Evidencia de aceptación |
| --- | --- | --- | --- | --- |
| Consulta de servicio: HU-16/17/21, RF-17/19, RN-08/11 | Capturar dato acordado; cancelar; validar; mostrar sólo resumen público; manejar ausencia y fallos | Definir identificación/verificación; comprobar propietario; devolver estado, equipo y actualización; reutilizar lógica de consulta | Integrar consulta por el mismo Backend; asegurar acceso móvil/escritorio previsto en HU-16, sin asumir que la interfaz de escritorio cubre móvil | El mismo servicio da el mismo resultado en consulta web y bot; servicio ajeno no revela datos |
| FAQ: HU-22 | Consultar catálogo Backend; responder pregunta conocida; reconocer ausencia de respuesta | Persistir preguntas y respuestas; API de lectura para bot y mantenimiento con permiso administrador | Pantalla para agregar y editar preguntas/respuestas; aplicar cambios mediante API | Un cambio administrativo se refleja en una nueva consulta sin editar TypeScript; pregunta desconocida no produce respuesta inventada |
| Aviso Listo: HU-18, RF-18, RN-09 | Enviar por Meta una solicitud autorizada y reportar resultado | Al pasar a Listo generar registro/evento; procesar envío; conservar resultado; evitar dependencia del cambio de estado respecto del canal | Actualizar estado mediante API; mostrar estado y resultado de notificación pertinente | Un cambio a Listo genera evento; Meta caído no revierte ni pierde el cambio |
| Equipo no recogido: HU-19, RN-10 | Enviar recordatorio solicitado por Backend | Guardar periodo configurable; detectar órdenes Listo vencidas; controlar duplicados | Formulario administrativo del periodo, con validación | Cambiar periodo altera elegibilidad; ejecutar proceso dos veces no duplica envíos sin control |
| Mantenimiento preventivo: HU-20, RN-10 | Enviar recordatorio solicitado por Backend | Programar desde Entregado; vincular cliente, equipo y servicio; aplicar periodo configurable | Formulario administrativo del periodo | No programa servicios sin entregar; registra referencias y fecha; no fija seis meses como constante obligatoria |
| Calidad: RNF-01–RNF-05 aplicables | Errores comprensibles; regresión; medir latencia | Permisos, hash seguro para cuentas del sistema, integridad, conservación ante fallos y respaldo/restauración | Validación de formularios, mensajes claros y permisos visibles | Evidencias de rendimiento, permisos, fallos e integridad; restauración de datos que soportan el módulo |

HU-16 menciona móvil/escritorio y HU-21 exige coincidencia con consulta web. Dado que el repositorio tiene Frontend de escritorio, la consulta web/móvil es una dependencia por resolver con el equipo: no debe declararse cubierta sólo por Tkinter.



## 4. Paquetes de trabajo en orden

### P1 — Acordar contratos y resolver arquitectura

- **Backend:** acordar formato de folio, campos públicos y mecanismo para verificar al cliente. HU-16 deja este mecanismo pendiente; teléfono verificado u OTP son opciones técnicas, no decisiones ya tomadas por los PDF.
- **Bot:** documentar entradas y salidas; reemplazar acceso PostgreSQL directo por adaptador HTTP, si se conservará historial; probar los métodos que necesita BuilderBot.
- **Frontend:** definir pantallas FAQ, periodos y consulta; precisar cómo cubrir consulta web/móvil.
- **Salida:** contratos revisables, diagrama aprobado por el equipo y criterios de prueba. No activar consulta real antes de completar propiedad y verificación.

### P2 — Consulta de reparación completa

- **Backend:** endpoint público autorizado para resumen, con identidad del bot y verificación del cliente; misma lógica que la consulta web.
- **Bot:** conectar flujo existente al endpoint real; validar folio y DTO; distinguir errores de infraestructura y consulta no verificada.
- **Frontend:** integrar consulta y datos mínimos autorizados, sin notas internas.
- **Salida:** conversación real por Meta y comprobación del mismo estado en ambos canales. El fixture DEMO-001 no satisface por sí solo RF-19.

### P3 — Evento de equipo listo

- **Backend:** persistir cambio de estado y evento de notificación de forma coherente; procesar envío separado del cambio; guardar fecha, canal, destinatario y resultado.
- **Bot:** servicio de envío Meta con acceso interno autenticado; devolver éxito o fallo al Backend.
- **Frontend:** mantener actualización de estado y presentar resultados pertinentes.
- **Salida:** orden permanece Listo incluso si el canal falla. El mecanismo de cola/outbox es una propuesta para lograrlo, no una tecnología exigida por el requisito.

### P4 — FAQ administrables

- **Backend:** modelo, validaciones y endpoints de mantenimiento; exigir administrador para cambios y exponer sólo contenido autorizado al bot.
- **Frontend:** listado y edición de preguntas/respuestas, con errores de guardado y permisos claros.
- **Bot:** flujo de consulta de FAQ, sin inventar respuestas ni abrir un editor de comportamientos.
- **Salida:** administrador cambia una respuesta y el bot usa el contenido actualizado en la siguiente consulta. Caché, si se usa, requiere actualización definida y verificable.

### P5 — Recordatorios configurables

- **Backend:** almacenar periodos; proceso programado para Listo no recogido y Entregado con mantenimiento pendiente; registrar programación y control de duplicados.
- **Frontend:** editar ambos periodos con rol administrador y unidades explícitas.
- **Bot:** reutilizar envío Meta; no implementar otra agenda independiente de la del Backend.
- **Salida:** pruebas con fechas ficticias y avance controlado del tiempo; registro de envíos sin esperar meses reales.

### P6 — Integración, calidad y entrega

- **Equipo:** pruebas E2E, permisos, fallos de Meta/Backend, rendimiento aproximado de dos segundos bajo carga normal (RNF-01), integridad y respaldo/restauración.
- **Bot:** revisar configuración Meta vigente y mecanismos necesarios para envíos proactivos; no asumir que un mensaje iniciado por el sistema se envía igual que una respuesta a consulta.
- **Salida:** notebook actualizado, manual de configuración, contratos, evidencias y defectos corregidos.



## 5. Calendario integrado

Las fechas y entregas siguen IEEE §3.2; la distribución detallada del trabajo del bot es una propuesta. Los responsables son los roles del equipo, no asignaciones individuales nuevas.

| Sprint 2026 | Bot | Backend | Frontend | Hito |
| --- | --- | --- | --- | --- |
| 4: 5–11 octubre | P1: contratos y diseño del adaptador HTTP | Identificación, privacidad y modelos necesarios | Diseño de consulta y administración | Arquitectura y contratos definidos |
| 5: 12–18 octubre | P2: integrar consulta real | Consulta autorizada y persistencia necesaria | Integrar consulta disponible y resolver canal web/móvil | Consulta integrada en entorno de pruebas |
| 6: 19–25 octubre | P3: envío Meta y errores | Evento Listo y registro de resultado | Estado y resultado del aviso | Pruebas del aviso y coherencia de consulta |
| 7: 26–30 octubre | Integración y regresión | Cerrar dependencias RF-17/18/19 | Corregir integración | Beta para entrega del 30 de octubre |
| 8: 2–8 noviembre | Robustez y envíos | P5: procesos de recordatorios y permisos | Formularios y permisos | Fallos controlados y recordatorios probados |
| 9: 9–15 noviembre | P4: FAQ y actualización de contenido | FAQ, configuración y respaldo/restauración | FAQ y periodos administrables | Versión candidata con alcance completo |
| 10: 16–22 noviembre | P6: latencia y regresión | Integridad, confiabilidad y evidencias | Usabilidad y correcciones | Evidencias de calidad |
| 11: 23–29 noviembre | Manual y demo E2E | Correcciones y despliegue | Aceptación de pantallas | Versión final para entrega del 30 de noviembre |

No se consideran completadas las entregas por el simple hecho de tener fechas en este plan. El avance se acredita con los criterios de aceptación.



## 6. Contratos propuestos entre módulos

Las rutas siguientes son propuestas de diseño; no existen todas en el repositorio ni los documentos obligan a usar estos nombres.

| Contrato | Consumidor | Permisos / responsabilidad |
| --- | --- | --- |
| Consulta de servicio y verificación | Bot y consulta web | Backend valida identidad y propiedad; retorna campos públicos |
| Lectura de FAQ | Bot | Sólo contenido autorizado del catálogo |
| Mantenimiento de FAQ | Frontend | Usuario administrador autenticado |
| Lectura/actualización de periodos | Frontend y proceso Backend | Administrador modifica; Backend aplica las reglas |
| Solicitud de envío / resultado | Backend y bot | Autenticación entre servicios; identificación del evento; control de duplicados |
| Persistencia conversacional, si se conserva | Adaptador HTTP del bot | Token de servicio restringido; Backend escribe PostgreSQL; no usar credenciales del administrador |



## 7. Flujos que se entregarán

```mermaid
flowchart TD
    A[Cliente consulta reparación] --> B[Bot captura dato acordado]
    B --> C[Backend verifica identidad y propiedad]
    C --> D{Autorizado}
    D -->|Sí| E[Resumen público]
    D -->|No| F[Respuesta genérica]
    G[Cliente pregunta FAQ] --> H[Bot consulta catálogo Backend]
    H --> I{Hay respuesta autorizada}
    I -->|Sí| J[Respuesta actualizada]
    I -->|No| K[Indicar que no hay respuesta]
```

```mermaid
sequenceDiagram
    actor Administrador
    participant Frontend
    participant Backend
    participant DB as PostgreSQL central
    participant Bot
    Administrador->>Frontend: Cambiar respuesta FAQ o periodo
    Frontend->>Backend: Petición con sesión del administrador
    Backend->>Backend: Validar rol y datos
    Backend->>DB: Guardar configuración
    Backend-->>Frontend: Confirmación
    Note over Backend,Bot: FAQ se consulta al atender pregunta; periodos los aplica Backend
    Bot->>Backend: Consultar FAQ para nueva pregunta
    Backend->>DB: Leer contenido vigente
    Backend-->>Bot: Respuesta autorizada
```



## 8. Pruebas y definición de terminado

| Prueba | Resultado esperado |
| --- | --- |
| Folio propio verificado | Estado, equipo y actualización correctos |
| Folio ajeno/inexistente | No expone existencia ni datos de terceros |
| Dato inválido/cancelación | Repite captura o termina de forma controlada |
| Cambio de FAQ por administrador | Nueva consulta usa la nueva respuesta |
| Edición FAQ/periodos por usuario sin permiso | Backend rechaza aunque se intente fuera del Frontend |
| Orden cambia a Listo con Meta caído | Orden conserva estado y queda registrado el fallo |
| Proceso de recordatorios repetido | No genera duplicados sin control |
| Servicio sin entregar | No genera recordatorio preventivo |
| Bot y consulta web | Resumen consistente de la misma orden |
| Caída Backend / reinicio del bot | Respuesta controlada y recuperación comprobada; no inventa información |
| Respaldo/restauración | Recupera configuración y registros relacionados según RNF-05 del IEEE |

Un paquete termina cuando pasan sus criterios E2E y están documentados contrato, configuración y limitaciones. Las pruebas unitarias del prototipo no acreditan por sí solas integración de Backend, Frontend o Meta.



## 9. Punto de partida y pendientes reales

Existen un flujo de consulta con demo/API, cliente HTTP con validación, configuración Meta, adaptador PostgreSQL directo y pruebas del prototipo. Los módulos actuales de Backend y Frontend se centran en autenticación, usuarios, clientes y equipos; esta revisión no encontró módulos de FAQ o recordatorios.

Pendientes: endpoint de consulta definitivo; verificación del cliente; adaptación de persistencia a Backend; catálogo FAQ; pantallas administrativas; registro/procesamiento de notificaciones; recordatorios; integración web/móvil; prueba real por Meta. Este plan no implementa esas funciones ni cambia el código de negocio.


# Contrato propuesto de consulta — pendiente de implementar

`POST {BACKEND_URL}/chatbot/servicios/consulta`, con `Authorization: Bearer <BACKEND_BOT_TOKEN>` y `Content-Type: application/json`.

Petición:

```json
{"folio":"FIX-001","telefono":"5213312345678"}
```

`telefono` proviene de `ctx.from` del proveedor Meta, no de un campo capturado. Backend valida el token del bot, normaliza el teléfono conforme al esquema acordado y verifica que pertenece al propietario del servicio. La comparación propuesta requiere un teléfono previamente verificado y no reemplaza un mecanismo adicional si el equipo lo exige. Antes de producción verificar autenticidad del webhook, límites de intentos y tratamiento de números reasignados.

Respuesta 200 (únicamente después de autorizar):

```json
{"folio":"FIX-001","estado":"En revisión","equipo":"Laptop Lenovo","actualizadoEn":"2026-10-04T12:00:00-06:00"}
```

| Código | Significado para el cliente HTTP |
| --- | --- |
| 200 | Validar DTO y mostrar sólo campos permitidos |
| 403 / 404 | Misma respuesta conversacional: no se pudo verificar el servicio |
| 401 | Credencial del bot inválida: consulta no disponible |
| 429 / 5xx | Consulta no disponible; sin reintentos automáticos |
| Timeout / red / JSON inválido | Consulta no disponible |

No retornar notas internas, domicilio, teléfonos ni datos de otros clientes. El bot rechaza estados desconocidos, folios distintos al solicitado y fechas inválidas. Backend es responsable de propiedad, límites de consulta y auditoría; nunca aceptar sólo el folio como autorización. Usar HTTPS fuera del entorno local.

No se ha implementado este endpoint ni modificado Backend en este ejemplo. Los mocks de pruebas sólo verifican el cliente HTTP, no demuestran autorización real.


# Guía del flujo modelo: consulta de reparación

Esta guía explica el código existente para que puedas construir los demás flujos. El plan está en `PLAN_CHATBOT.ipynb`; el contrato HTTP propuesto, en `CONTRATO_API.ipynb`. Las referencias a HU/RF y fechas de ese plan están pendientes de contrastar nuevamente: los tres PDF originales no están disponibles en la nueva ruta del proyecto.



## Recorrido de la conversación

```mermaid
flowchart TD
    A[Cliente escribe estado] --> B[Bot solicita folio]
    B --> C{Respuesta}
    C -->|cancelar| D[Termina sin consultar]
    C -->|Formato inválido| E[Explica formato y vuelve a capturar]
    E --> C
    C -->|Folio válido| F[Consulta servicio con folio y remitente Meta]
    F --> G{Resultado}
    G -->|encontrado| H[Muestra campos públicos y termina]
    G -->|no_autorizado| I[Respuesta genérica y termina]
    G -->|no_disponible| J[Informa indisponibilidad y termina]
```



## Cómo leer cada archivo

1. **`src/config.ts`**: `requerida` comprueba variables sin revelar valores. `cargarConfiguracion` valida puerto y modo, prepara opciones de Meta y exige credenciales de Backend sólo en modo API. No configura la aplicación de Meta: eso se hace en su panel.
2. **`src/app.ts`**: selecciona el servicio demo o HTTP, crea `MetaProvider`, registra el flujo con `createFlow` y arranca el servidor. Es el lugar donde conectarás un flujo nuevo. `PostgreSQLAdapter` persiste eventos y contactos; la información real del servicio pertenece al Backend.
3. **`src/flows/consulta.flow.ts`**: `crearFlujoConsulta` recibe la función `consultar` y la bandera demo. `addKeyword` define entradas; `addAnswer` muestra la pregunta y `capture: true` espera la respuesta. El callback usa `ctx.body` para el folio y `ctx.from` para el remitente. Primero permite cancelar, después valida y finalmente consulta. `fallBack` vuelve a pedir el dato; `endFlow` envía la respuesta final. Las opciones están descritas en la [documentación de funciones de BuilderBot](https://www.builderbot.app/en/add-functions).
4. **`src/services/consultar-servicio.ts`**: define estados y datos públicos, normaliza el folio y valida respuestas externas. `crearConsultaApi` envía folio y teléfono con token del bot y timeout de cinco segundos. `leerServicio` selecciona los campos que se pueden mostrar. `crearConsultaDemo` devuelve un caso ficticio únicamente para el remitente configurado.
5. **`tests/consulta.test.ts`**: comprueba folios, selección de campos, restricción demo y manejo del transporte HTTP. Los mocks no verifican autorización real del Backend ni entrega por WhatsApp.



## Ejemplo para probar

Con `CHATBOT_MODE=demo`, variables de Meta completas y `DEMO_PHONE_NUMBER` configurado:

```text
Cliente: estado
Bot: Bienvenido a Fixify... Escribe tu folio o cancelar.
Cliente: DEMO-001
Bot: DATOS FICTICIOS — DEMOSTRACIÓN
     Folio: DEMO-001
     Equipo: Laptop de demostración
     Estado: En revisión
     Última actualización: 2026-10-04T12:00:00-06:00
```

Comprueba también `AB` (repite captura), `cancelar` (termina) y un folio diferente (respuesta genérica). En modo API comprueba Backend caído, respuesta inválida y servicio ajeno. El endpoint propuesto todavía requiere implementación e integración; no se considera finalizada la consulta real.



## Patrón para tu siguiente flujo

Antes de escribirlo, completa esta ficha:

| Parte | Qué documentar |
| --- | --- |
| Requisito | HU/RF y página del documento que lo sustenta |
| Entrada | Palabras que activan el flujo y datos solicitados |
| Validación | Formato, cancelación, reintentos y caducidad |
| Servicio | Contrato, identidad, permisos y respuesta pública |
| Salida | Mensajes de éxito, dato desconocido y fallo técnico |
| Evidencia | Conversación y pruebas; limitaciones pendientes |

Luego crea un servicio tipado y una fábrica de flujo en archivos separados. Inyecta el servicio desde `app.ts` y registra el flujo junto a `crearFlujoConsulta(...)` en `createFlow`. Usa palabras de entrada distintas. Si capturas varios datos, guarda sólo los necesarios en `state`, valida cada paso y elimina los datos al cancelar o concluir. Sigue el mismo orden: capturar → validar → consultar → responder → cerrar.

Documenta cada función con su propósito, entradas, resultado, errores y requisito asociado. En los comentarios interiores explica decisiones, por ejemplo por qué folio inexistente y servicio ajeno reciben el mismo mensaje. Evita duplicar en comentarios lo que ya expresa una línea de código.



## Comprobación del entorno

Ejecuta `pnpm test`, `pnpm lint` y `pnpm build` desde `chatbot/base-ts-meta-postgres`. Si los lanzadores buscan `node.exe` en Linux, las dependencias fueron copiadas desde otro sistema; reinstálalas en Linux con `pnpm install --frozen-lockfile`. No cambies el código del flujo para corregir ese problema de instalación.

La configuración del proveedor y webhook se describe en la [guía oficial de Meta en BuilderBot](https://www.builderbot.app/en/providers/meta). No uses sus ejemplos antiguos de versión o tarifas como configuración vigente de tu cuenta.


## Ejecución del flujo, decisión por decisión

1. **Activación**: el mensaje `estado` coincide con `addKeyword`. No consulta la API todavía.
2. **Pregunta**: `addAnswer` presenta la solicitud; `capture: true` entrega la siguiente respuesta al callback.
3. **Cancelación**: si responde `cancelar`, `endFlow` termina sin llamada HTTP. La persistencia del adaptador puede guardar eventos; cancelar no borra el historial.
4. **Validación**: `validarFolio` limpia espacios y convierte a mayúsculas. `AB` no pasa; `fallBack` vuelve a pedir un folio. Un formato válido no demuestra que exista ni que pertenezca al remitente.
5. **Consulta**: `consultar(folio, ctx.from)` usa el remitente recibido, sin pedirlo al usuario. En demo usa el fixture; en API hace una petición al Backend. La contraseña PostgreSQL no se envía al Backend.
6. **Clasificación**: `no_autorizado` oculta si la orden existe; `no_disponible` evita inventar un estado cuando falla el sistema; `encontrado` contiene el DTO público validado.
7. **Respuesta final**: `endFlow` muestra sólo folio, equipo, estado y fecha. Para consultar de nuevo, el cliente envía `estado` otra vez.

| Mensaje / evento | Acción del bot | ¿Llama al Backend? |
| --- | --- | --- |
| `estado` | Pregunta folio | No |
| `AB` | Repite captura | No |
| `cancelar` | Finaliza | No |
| `FIX-001` en modo API | Consulta con remitente | Sí |
| Backend 403/404 | Finaliza con respuesta genérica | Ya se llamó |
| Backend 401/500/timeout | Finaliza con indisponibilidad | Ya se llamó |

`state` no se utiliza en este flujo de una captura. PostgreSQL guarda eventos y permite buscar la última entrada; no garantiza persistencia de todos los temporizadores, estados en memoria o capturas pendientes del motor. Prueba reinicios antes de prometer continuidad de un diálogo de varios pasos.

## Cómo ampliar a un flujo de varios pasos

```mermaid
flowchart TD
    A[Activar flujo] --> B[Capturar primer dato]
    B --> C{Cancelar o dato válido}
    C -->|cancelar| X[Limpiar datos propios y terminar]
    C -->|inválido| B
    C -->|válido| D[Guardar dato mínimo en state]
    D --> E[Capturar segundo dato]
    E --> F{Validar o cancelar}
    F -->|cancelar| X
    F -->|inválido| E
    F -->|válido| G[Llamar servicio Backend]
    G --> H[Responder, limpiar datos propios y cerrar]
```

Para FAQ puedes mantener una sola captura: pregunta → búsqueda en catálogo autorizado → respuesta o derivación a recepción. Para notificaciones de equipo listo, la entrada es un evento del Backend y el envío requiere autorización del canal; no copies el flujo de captura como si fuera una notificación automática.

Lee [Autenticación y PostgreSQL](../../chatbot/docs/AUTENTICACION_Y_POSTGRESQL.ipynb) para entender por qué el folio y el teléfono tienen funciones distintas.


# Autenticación y autorización: explicación paso a paso

Autenticar significa comprobar quién hace una petición. Autorizar significa comprobar qué puede consultar esa identidad. Que el bot tenga un token válido no permite mostrarle cualquier reparación a cualquier cliente.



## Qué significa cada credencial

| Dato | Quién lo usa | Qué demuestra | Qué no demuestra |
| --- | --- | --- | --- |
| `META_ACCESS_TOKEN` | Bot ante Meta | Permiso para usar la API de WhatsApp de la cuenta | Que un cliente sea dueño de una orden |
| `META_PHONE_NUMBER_ID` | Bot ante Meta | Identifica el número empresarial | No es una contraseña ni el teléfono del cliente |
| `META_VERIFY_TOKEN` | Meta y webhook durante su alta GET | Que el endpoint responde al reto configurado | No valida la firma de cada mensaje POST |
| Usuario/contraseña PostgreSQL | Bot ante PostgreSQL | Permiso de conexión a la base de conversaciones | No verifica al cliente de WhatsApp |
| `BACKEND_BOT_TOKEN` | Bot ante Backend | Identidad del servicio chatbot | No concede automáticamente acceso a una orden ajena |
| `ctx.from` | Flujo y Backend | Remitente entregado por el proveedor | Por sí solo no prueba propiedad sin verificar webhook y vinculación previa |
| Folio | Cliente y Backend | Identifica el servicio solicitado | No es una contraseña ni autorización |

El nombre interno `jwtToken` del proveedor recibe `META_ACCESS_TOKEN`: no implica que debas generar un JWT propio para Meta. El token del Backend es otra credencial; no reutilices el de Meta. El contrato propone Bearer, pero el Backend aún debe implementar su validación y permisos.



## Ejemplo concreto

Ana tiene la orden FIX-001 y un teléfono previamente verificado en recepción. Bruno conoce ese folio. Ambos escriben al bot. El token del bot será el mismo en las dos peticiones, porque identifica a la aplicación. Lo que cambia es el remitente `ctx.from`. El Backend debe comprobar que el remitente corresponde al propietario de FIX-001; sólo Ana recibe el estado. Bruno recibe el mismo mensaje que recibiría con un folio inexistente.

No pidas al cliente que escriba su teléfono como prueba de identidad: podría escribir el de otra persona. No envíes contraseñas del personal por WhatsApp. Vincular un teléfono previamente verificado es una propuesta del ejemplo, pendiente de aprobación funcional con el equipo; si se requiere OTP, debes implementar generación, caducidad, límites de intentos y validación en Backend. El ejemplo actual no implementa OTP.



## Secuencia de consulta autorizada

```mermaid
sequenceDiagram
    actor Cliente
    participant Meta
    participant Bot
    participant Backend
    participant Negocio as PostgreSQL negocio
    Cliente->>Meta: estado y luego folio
    Meta->>Bot: webhook con remitente
    Note over Meta,Bot: Verificar autenticidad del POST antes de confiar en remitente
    Bot->>Bot: validar formato del folio
    Bot->>Backend: POST folio + ctx.from + Bearer token del bot
    Backend->>Backend: autenticar token del bot
    Backend->>Negocio: buscar orden y propietario
    Backend->>Backend: comprobar teléfono verificado y propiedad
    alt Propietario autorizado
        Backend-->>Bot: 200 con campos públicos
        Bot-->>Meta: estado público
        Meta-->>Cliente: respuesta
    else Folio ajeno o inexistente
        Backend-->>Bot: 403 o 404
        Bot-->>Meta: mismo mensaje genérico
        Meta-->>Cliente: no se pudo verificar
    else Credencial del bot inválida
        Backend-->>Bot: 401
        Bot-->>Meta: consulta no disponible
    end
```



## Qué hace hoy el código y qué falta

El flujo toma `ctx.from` y el cliente HTTP lo envía con el folio. El cliente HTTP agrega `Authorization: Bearer ...`, selecciona campos públicos y convierte errores en respuestas controladas. En demo sólo se compara con `DEMO_PHONE_NUMBER`: es un dato ficticio para practicar, no autenticación productiva.

El endpoint del contrato todavía no existe. Faltan autenticación del bot, autorización por propietario, normalización acordada del teléfono, límites de intentos y auditoría en Backend. El código inspeccionado de MetaProvider 1.4.2 registra el reto GET y el receptor POST; no se debe dar por implementada la validación de firma del POST. Antes de activar consultas reales, verificar la firma de Meta con el secreto de la aplicación sobre el cuerpo original en un middleware compatible o un gateway. Esa implementación queda pendiente; `META_VERIFY_TOKEN` no la reemplaza.



## Dónde se guardan los datos

```mermaid
flowchart LR
    W[WhatsApp cliente] --> M[Meta]
    M --> F[Flujo BuilderBot]
    F --> H[(PostgreSQL chatbot: history y contact)]
    F --> A[Backend: autentica y autoriza]
    A --> N[(PostgreSQL negocio: clientes y órdenes)]
```

El adaptador persiste respuestas y eventos con teléfonos. El historial contiene información del cliente y exige control de acceso y retención. Las tablas del adaptador no son las tablas `clientes` y `ordenes_servicio`. Usa `fixify_chatbot` como base dedicada para evitar colisiones y permisos sobre el negocio. No escribas tokens en mensajes ni en los datos guardados por el flujo.

Fuentes: [adaptadores de BuilderBot](https://www.builderbot.app/en/databases), [proveedor Meta](https://www.builderbot.app/en/providers/meta). Diagramas en celdas Markdown con bloques Mermaid; utiliza un visor de notebooks que soporte Mermaid, como JupyterLab moderno. Si el visor sólo muestra texto, abre el notebook en JupyterLab o habilita soporte Mermaid.


## Código comentado: src/config.ts

```typescript
/** Lee variables obligatorias sin incluir sus valores en mensajes de error. */
const requerida = (nombre: string): string => {
    const valor = process.env[nombre]?.trim()
    if (!valor) throw new Error(`Falta configurar ${nombre}`)
    return valor
}

/** Falla antes de iniciar si la configuración es incompleta; API es el modo predeterminado. */
export const cargarConfiguracion = () => {
    const modo = process.env.CHATBOT_MODE ?? 'api'
    if (!['api', 'demo'].includes(modo)) throw new Error('CHATBOT_MODE debe ser api o demo')
    const port = Number(process.env.PORT ?? 3008)
    if (!Number.isInteger(port) || port < 1 || port > 65535) throw new Error('PORT inválido')
    const dbPort = Number(process.env.POSTGRES_PORT ?? 5432)
    if (!Number.isInteger(dbPort) || dbPort < 1 || dbPort > 65535) throw new Error('POSTGRES_PORT inválido')
    const postgres = {
        host: requerida('POSTGRES_HOST'), user: requerida('POSTGRES_USER'),
        password: requerida('POSTGRES_PASSWORD'), database: requerida('POSTGRES_DB'), port: dbPort,
    }
    const meta = {
        jwtToken: requerida('META_ACCESS_TOKEN'), numberId: requerida('META_PHONE_NUMBER_ID'),
        verifyToken: requerida('META_VERIFY_TOKEN'), version: requerida('META_API_VERSION'),
    }
    const apiUrl = modo === 'api' ? requerida('BACKEND_URL') : ''
    if (modo === 'api' && !['http:', 'https:'].includes(new URL(apiUrl).protocol)) throw new Error('BACKEND_URL inválida')
    return { port, meta, postgres, demo: modo === 'demo', apiUrl,
        apiToken: modo === 'api' ? requerida('BACKEND_BOT_TOKEN') : '',
        telefonoDemo: modo === 'demo' ? requerida('DEMO_PHONE_NUMBER') : '',
    }
}

```


## Código comentado: src/database/postgres.ts

```typescript
import { PostgreSQLAdapter } from '@builderbot/database-postgres'

/** Configuración de la base de eventos/contactos de BuilderBot, separada del negocio. */
export interface ConfiguracionPostgres {
    host: string
    user: string
    password: string
    database: string
    port: number
}

/**
 * El adaptador 1.4.2 llama init desde su constructor sin esperar las tablas.
 * ready permite esperar también su creación antes de abrir el webhook.
 * Se resuelve false ante un fallo para no dejar rechazos sin manejar en el constructor.
 */
class BaseConversaciones extends PostgreSQLAdapter {
    declare ready: Promise<boolean>

    override init(): Promise<boolean> {
        this.ready = (async () => {
            try {
                await super.init()
                // El init original inicia esta operación sin esperarla.
                // Capturamos esa operación mediante el método sobrescrito de abajo.
                await this.tablasListas
                return true
            } catch {
                return false
            }
        })()
        return this.ready
    }

    declare tablasListas: Promise<void>

    override checkTableExistsAndSP(): Promise<void> {
        this.tablasListas = super.checkTableExistsAndSP()
        // El init del paquete no observa este rechazo; ready sí lo comprueba.
        void this.tablasListas.catch(() => undefined)
        return this.tablasListas
    }
}

/** No inicia Meta si PostgreSQL no está disponible; nunca cambia a memoria automáticamente. */
export const crearBaseConversaciones = async (config: ConfiguracionPostgres): Promise<PostgreSQLAdapter> => {
    const database = new BaseConversaciones(config)
    if (!await database.ready) throw new Error('No se pudo preparar PostgreSQL para el chatbot')
    return database
}

```


## Código comentado: src/app.ts

```typescript
import { createBot, createProvider, createFlow } from '@builderbot/bot'
import { crearBaseConversaciones } from './database/postgres'
import { MetaProvider } from '@builderbot/provider-meta'
import { cargarConfiguracion } from './config'
import { crearFlujoConsulta } from './flows/consulta.flow'
import { crearConsultaApi, crearConsultaDemo } from './services/consultar-servicio'

/**
 * Punto de composición: configura Meta y conecta los flujos con sus servicios.
 * PostgreSQL persiste eventos y contactos del bot en una base dedicada.
 * Las órdenes del negocio siguen consultándose por Backend, con autorización.
 */
const main = async () => {
    const config = cargarConfiguracion()
    const consultar = config.demo
        ? crearConsultaDemo(config.telefonoDemo)
        : crearConsultaApi(config.apiUrl, config.apiToken)
    // Espera conexión y tablas antes de recibir mensajes de Meta.
    const database = await crearBaseConversaciones(config.postgres)
    const provider = createProvider(MetaProvider, config.meta)
    const { httpServer } = await createBot({
        flow: createFlow([crearFlujoConsulta(consultar, config.demo)]),
        provider, database,
    })
    // El proveedor registra el webhook de Meta. No se exponen las rutas administrativas de la plantilla.
    httpServer(config.port)
}

// Una configuración inválida termina el proceso para evitar un bot parcialmente iniciado.
main().catch((error: unknown) => {
    console.error(error instanceof Error ? error.message : 'No se pudo iniciar el chatbot')
    process.exitCode = 1
})

```


## Código comentado: src/flows/consulta.flow.ts

```typescript
import { addKeyword } from '@builderbot/bot'
import { PostgreSQLAdapter } from '@builderbot/database-postgres'
import { MetaProvider } from '@builderbot/provider-meta'
import { ConsultarServicio, validarFolio } from '../services/consultar-servicio'

/**
 * Flujo modelo: HU-21 / RF-19, apoyado por HU-16/17 y RF-17.
 * Entrada -> captura de folio -> validación -> Backend -> respuesta pública -> cierre.
 * La dependencia se inyecta para usar el mismo diálogo en demo y con API real.
 */
export const crearFlujoConsulta = (consultar: ConsultarServicio, demo: boolean) =>
    addKeyword<MetaProvider, PostgreSQLAdapter>(['hola', 'menu', 'menú', 'estado', 'consulta'], { sensitive: true })
        .addAnswer([
            'Bienvenido a Fixify. Aquí puedes consultar el estado de tu reparación.',
            ...(demo ? ['MODO DEMOSTRACIÓN: sólo datos ficticios. Folio de ejemplo: DEMO-001.'] : []),
            'Escribe tu folio o *cancelar* para salir.',
        ], { capture: true }, async (ctx, { fallBack, endFlow }) => {
            // Cancelar no consulta la API ni conserva información adicional en estado.
            if (ctx.body.trim().toLowerCase() === 'cancelar') return endFlow('Consulta cancelada. Escribe *estado* para empezar de nuevo.')
            const folio = validarFolio(ctx.body)
            // fallBack mantiene la captura en este paso hasta recibir un folio válido.
            if (!folio) return fallBack('Usa un folio de 3 a 40 caracteres: letras, números o guiones. También puedes escribir *cancelar*.')
            const resultado = await consultar(folio, ctx.from)
            if (resultado.tipo === 'no_autorizado') {
                return endFlow('No pudimos verificar un servicio con esos datos. Revisa tu folio o contacta a recepción. Escribe *estado* para reintentar.')
            }
            if (resultado.tipo === 'no_disponible') {
                return endFlow('La consulta no está disponible en este momento. Intenta después o contacta a recepción.')
            }
            const s = resultado.servicio
            // Se muestran exclusivamente los campos del DTO; no se vuelca el JSON del Backend.
            return endFlow([
                ...(demo ? ['DATOS FICTICIOS — DEMOSTRACIÓN'] : []),
                `Folio: ${s.folio}`, `Equipo: ${s.equipo}`, `Estado: ${s.estado}`,
                `Última actualización: ${s.actualizadoEn}`,
                'Escribe *estado* para otra consulta.',
            ].join('\n'))
        })

```


## Código comentado: src/services/consultar-servicio.ts

```typescript
/** Estados oficiales: RN-03. El bot sólo presenta datos; no cambia órdenes. */
export const ESTADOS = ['Recibido', 'En revisión', 'Reparando', 'Listo', 'Entregado'] as const
export type EstadoServicio = typeof ESTADOS[number]

/** DTO público propuesto: excluye notas técnicas, costos internos y datos personales. */
export interface ServicioPublico {
    folio: string
    estado: EstadoServicio
    equipo: string
    actualizadoEn: string
}
export type ResultadoConsulta =
    | { tipo: 'encontrado'; servicio: ServicioPublico }
    | { tipo: 'no_autorizado' }
    | { tipo: 'no_disponible' }
export type ConsultarServicio = (folio: string, telefono: string) => Promise<ResultadoConsulta>

/** Formato provisional, pendiente de acordar con Backend; no es un requisito del PDF. */
export const validarFolio = (texto: string): string | null => {
    const folio = texto.trim().toUpperCase()
    return /^[A-Z0-9][A-Z0-9-]{2,39}$/.test(folio) ? folio : null
}

/** Valida la respuesta externa y selecciona únicamente los campos públicos permitidos. */
export const leerServicio = (dato: unknown, folio: string): ServicioPublico | null => {
    if (!dato || typeof dato !== 'object') return null
    const s = dato as Record<string, unknown>
    if (s.folio !== folio || !ESTADOS.includes(s.estado as EstadoServicio) ||
        typeof s.equipo !== 'string' || !s.equipo.trim() || s.equipo.length > 160 ||
        typeof s.actualizadoEn !== 'string' || !Number.isFinite(Date.parse(s.actualizadoEn))) return null
    return { folio, estado: s.estado as EstadoServicio, equipo: s.equipo, actualizadoEn: s.actualizadoEn }
}

/**
 * Adaptador HTTP del contrato PROPUESTO (el endpoint aún no existe en Backend).
 * El teléfono proviene de ctx.from, nunca de una respuesta escrita por el cliente.
 * Backend debe autenticar al bot Y comprobar la propiedad del servicio (RN-08/RN-11).
 * Fallos, respuestas inválidas y timeout no generan estados inventados.
 */
export const crearConsultaApi = (url: string, token: string): ConsultarServicio => async (folio, telefono) => {
    try {
        const response = await fetch(new URL('chatbot/servicios/consulta', url.endsWith('/') ? url : `${url}/`), {
            method: 'POST',
            headers: { 'Content-Type': 'application/json', Authorization: `Bearer ${token}` },
            body: JSON.stringify({ folio, telefono }),
            signal: AbortSignal.timeout(5000),
            redirect: 'error',
        })
        // No revelar si el folio existe cuando la propiedad no coincide.
        if ([403, 404].includes(response.status)) return { tipo: 'no_autorizado' }
        if (!response.ok) return { tipo: 'no_disponible' }
        const servicio = leerServicio(await response.json(), folio)
        return servicio ? { tipo: 'encontrado', servicio } : { tipo: 'no_disponible' }
    } catch {
        // No registrar teléfonos, credenciales ni el contenido de la respuesta externa.
        return { tipo: 'no_disponible' }
    }
}

/** Fixture educativo, separado de producción; sólo acepta el remitente demo configurado. */
export const crearConsultaDemo = (telefonoDemo: string): ConsultarServicio => async (folio, telefono) => {
    if (folio !== 'DEMO-001' || telefono !== telefonoDemo) return { tipo: 'no_autorizado' }
    return { tipo: 'encontrado', servicio: {
        folio: 'DEMO-001', estado: 'En revisión', equipo: 'Laptop de demostración',
        actualizadoEn: '2026-10-04T12:00:00-06:00',
    } }
}

```


## Código comentado: tests/consulta.test.ts

```typescript
import { test } from 'node:test'
import assert from 'node:assert/strict'
import { crearConsultaApi, crearConsultaDemo, leerServicio, validarFolio } from '../src/services/consultar-servicio'
const publico = { folio: 'FIX-001', estado: 'En revisión', equipo: 'Laptop', actualizadoEn: '2026-10-04T12:00:00-06:00' }

test('normaliza folios y rechaza entradas inválidas', () => {
    assert.equal(validarFolio(' fix-001 '), 'FIX-001')
    for (const f of ['', 'AB', '../orden', 'A'.repeat(41)]) assert.equal(validarFolio(f), null)
})
test('el DTO elimina información privada y rechaza inconsistencias', () => {
    assert.deepEqual(leerServicio({ ...publico, notaInterna: 'privado' }, 'FIX-001'), publico)
    for (const s of [null, { ...publico, estado: 'Inventado' }, { ...publico, actualizadoEn: 'ayer' }, { ...publico, folio: 'OTRO' }]) {
        assert.equal(leerServicio(s, 'FIX-001'), null)
    }
})
test('demo restringe folio y remitente', async () => {
    const consultar = crearConsultaDemo('5213300000000')
    assert.equal((await consultar('DEMO-001', '5213300000000')).tipo, 'encontrado')
    assert.equal((await consultar('DEMO-001', 'otro')).tipo, 'no_autorizado')
    assert.equal((await consultar('OTRO', '5213300000000')).tipo, 'no_autorizado')
})
// Mock del transporte: verifica el cliente, no la autorización del Backend real.
test('API envía identidad/token y controla respuestas inválidas y fallos', async (t) => {
    const original = globalThis.fetch
    t.after(() => { globalThis.fetch = original })
    const consultar = crearConsultaApi('http://localhost/api/v1', 'token-prueba')
    globalThis.fetch = async (url, options) => {
        assert.equal(String(url), 'http://localhost/api/v1/chatbot/servicios/consulta')
        assert.deepEqual(JSON.parse(options.body as string), { folio: 'FIX-001', telefono: 'remitente-meta' })
        assert.equal((options.headers as Record<string, string>).Authorization, 'Bearer token-prueba')
        assert.ok(options.signal)
        return new Response(JSON.stringify({ ...publico, notaInterna: 'privado' }))
    }
    assert.deepEqual(await consultar('FIX-001', 'remitente-meta'), { tipo: 'encontrado', servicio: publico })
    for (const status of [403, 404, 401, 429, 500]) {
        globalThis.fetch = async () => new Response('', { status })
        assert.equal((await consultar('FIX-001', 'remitente-meta')).tipo,
            [403, 404].includes(status) ? 'no_autorizado' : 'no_disponible')
    }
    for (const body of ['no es JSON', JSON.stringify({ ...publico, estado: 'Inventado' })]) {
        globalThis.fetch = async () => new Response(body)
        assert.equal((await consultar('FIX-001', 'remitente-meta')).tipo, 'no_disponible')
    }
    globalThis.fetch = async () => { throw new DOMException('Timeout', 'TimeoutError') }
    assert.equal((await consultar('FIX-001', 'remitente-meta')).tipo, 'no_disponible')
})

```


## Validación de esta versión

Se verificó con PostgreSQL temporal, puerto 55439 y datos ficticios: creación de tablas/funciones y recuperación de un evento y contacto desde una segunda conexión. Pasaron cinco pruebas, ESLint, compilación y revisión de tipos. La instancia temporal se cerró al terminar. No se modificó la base del negocio ni se probó el webhook real de Meta.

`pnpm test` ejecuta las pruebas de consulta y omite la integración PostgreSQL por defecto. Para ejecutar integración usa una base dedicada de pruebas con `CHATBOT_TEST_POSTGRES=1`, `PG_TEST_HOST`, `PG_TEST_PORT`, `PG_TEST_USER`, `PG_TEST_PASSWORD` y `PG_TEST_DATABASE`. El test crea tablas y funciones del adaptador y elimina sus propios registros al terminar; no lo ejecutes contra la base productiva.


## Prueba de persistencia

```typescript
import { test } from 'node:test'
import assert from 'node:assert/strict'
import { crearBaseConversaciones } from '../src/database/postgres'

/** Opt-in: usa sólo una base de pruebas dedicada; el adaptador crea tablas y funciones. */
test('PostgreSQL persiste eventos entre dos conexiones independientes', {
    skip: process.env.CHATBOT_TEST_POSTGRES !== '1',
}, async (t) => {
    const config = {
        host: process.env.PG_TEST_HOST ?? '127.0.0.1',
        port: Number(process.env.PG_TEST_PORT ?? 55439),
        user: process.env.PG_TEST_USER ?? 'deadtales',
        password: process.env.PG_TEST_PASSWORD ?? 'prueba-local',
        database: process.env.PG_TEST_DATABASE ?? 'postgres',
    }
    const primera = await crearBaseConversaciones(config)
    t.after(() => primera.db.release(true))
    const from = `test-${Date.now()}`
    // Datos ficticios; el remitente único evita mezclar ejecuciones.
    await primera.save({ ref: 'test', keyword: 'estado', answer: 'Respuesta ficticia',
        refSerialize: 'test-ref', from, options: {} })
    const segunda = await crearBaseConversaciones(config)
    t.after(async () => {
        await segunda.db.query('DELETE FROM history WHERE phone = $1', [from])
        await segunda.db.query('DELETE FROM contact WHERE phone = $1', [from])
        segunda.db.release(true)
    })
    assert.equal((await segunda.getPrevByNumber(from))?.answer, 'Respuesta ficticia')
    assert.equal((await segunda.getContact({ from } as never))?.phone, from)
})

```
